# Walk Attention — a theory-first tutorial

Path algebras of quivers as a sparse multi-hop attention for mitigating over-squashing in GNNs. This single notebook walks the whole idea from scratch, with hand-drawn theoretical figures and runnable `aiq-quivers` / `networkx` code. Read top to bottom.

## Contents
0. Part 0 — Quivers, paths, and multiplicity
1. Part 1 — What is over-squashing?
2. Part 2 — The path algebra: counting paths
3. Part 3 — Paths are messages
4. Part 4 — The walk operator is attention
5. Part 5 — Putting it together

# P0 — Quivers, paths, and multiplicity

Everything in this project rests on one object: a **quiver** (a directed graph) and the **paths** in it. This
notebook builds the vocabulary with hand-drawn diagrams, and lets you *build the graphs yourself* with the
`aiq-quivers` library.

## 1. What is a quiver?

<img src="../figs-theory/en/T1_quiver_anatomy.svg" width="760"/>

In [ ]:
# Build that quiver with aiq-quivers. Arrows are (name, source, target) triples.
from aiq.quiver import Quiver
Q = Quiver([1, 2, 3], [('a', 1, 2), ('aprime', 1, 2), ('b', 2, 3), ('c', 3, 3)])
print('vertices Q0:', Q.Q0)
print('adjacency matrix A (A[i,j] = #arrows i->j):')
print(Q.adjacency_matrix().astype(int))

## 2. What is a path?

A path is a directed walk: arrows joined head-to-tail. We read them right-to-left, like composition.

<img src="../figs-theory/en/E0a_what_is_a_path.svg" width="760"/>

In [ ]:
# Let aiq-quivers enumerate the paths for us.
from aiq.path_algebra import PathAlgebra
line = Quiver([1,2,3,4], [('a1',1,2),('a2',2,3),('a3',3,4)])
paths = PathAlgebra(line).paths_from_to(1, 4, 3)
print('paths of length 3 from 1 to 4:', [str(p) for p in paths])

## 3. Walks may repeat — and that is what we count

<img src="../figs-theory/en/E0c_path_vs_walk.svg" width="760"/>

## 4. Multiplicity: 1 vs 2 vs 4 parallel paths

How many paths connect two nodes? That number — the **multiplicity** — is what drives over-squashing (P1).

<img src="../figs-theory/en/E0b_multiplicity_1_2_4.svg" width="760"/>

In [ ]:
# See it concretely: build the three graphs and count the source->target paths.
import numpy as np
from oversquash import viz

graphs = {
    'line (n=1)':    ([(0,1),(1,2)], 3),
    'diamond (n=2)': ([(0,1),(0,2),(1,3),(2,3)], 4),
    'three (n=3)':   ([(0,1),(0,2),(0,3),(1,4),(2,4),(3,4)], 5),
}
for name, (edges, n) in graphs.items():
    Qi = Quiver(list(range(n)), [(f'e{k}', u, v) for k,(u,v) in enumerate(edges)])
    A = Qi.adjacency_matrix(); g = 2
    count = int(np.linalg.matrix_power(A, g)[0, n-1])
    print(f'{name:16s}: {count} path(s) of length {g} from 0 to {n-1}')

In [ ]:
# Draw the diamond to see it (networkx).
viz.draw_graph([(0,1),(0,2),(1,3),(2,3)], 4, src=0, dst=3,
               pos={0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}, title='the diamond: 2 parallel paths 0->3')

**Next (P1):** more parallel same-length paths means more messages squashed together — over-squashing.

---

# P1 — What is over-squashing?

A message-passing GNN updates each node by mixing in its neighbours, one hop at a time. When many paths
funnel into a node, all their messages are crushed into one fixed-width vector and signal is lost. We make
this precise on a **bottleneck** graph you can build with `aiq-quivers`.

## 1. Message passing moves one hop per layer

To reach a node `g` hops away you need `g` layers — and everything along the way is re-squashed at each step.

<img src="../figs-theory/en/E1a_message_one_hop.svg" width="760"/>

## 2. The bottleneck and the formula K·M^d

<img src="../figs-theory/en/T2_bottleneck_formula.svg" width="760"/>

In [ ]:
# Build the bottleneck with aiq-quivers' helper and look at it.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash import viz
K, M = 4, 3
data = make_bottleneck_graph(K, M, depth=2, generator=torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='K sources -> d layers of M -> target')

## 3. A fixed-width vector overflows

The walk count into the target grows like `K·M^d`, but the target's vector stays the same size.

<img src="../figs-theory/en/E1b_vector_overflow.svg" width="760"/>

In [ ]:
# Count the messages forced into the target, for growing depth (via walk operators).
from oversquash.ideal_bridge import walk_operators
for d in [1, 2, 3, 4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'depth {d}: {int(raw[d][:, t].sum()):4d} messages squashed into one vector  (~ K*M^d)')

## 4. The walk count explodes with distance

<img src="../figs-theory/en/E1c_explosion.svg" width="760"/>

**Next (P2):** to reason about all these paths exactly, we need the path algebra `kQ`.

---

# P2 — The path algebra: counting paths with linear algebra

The number of paths between two nodes is exactly what gets squashed. The **path algebra** `kQ` makes that
number algebraic and computable: it equals an entry of a power of the adjacency matrix.

## 1. The adjacency matrix encodes the arrows

<img src="../figs-theory/en/E2a_adjacency_matrix.svg" width="760"/>

In [ ]:
# The diamond quiver and its adjacency matrix (aiq-quivers).
from aiq.quiver import Quiver
Q = Quiver([1,2,3,4], [('a1',1,2),('a2',1,3),('b1',2,4),('b2',3,4)])
A = Q.adjacency_matrix().astype(int)
print('A =\n', A)

## 2. A² counts length-2 paths

<img src="../figs-theory/en/E2b_A_squared.svg" width="760"/>

In [ ]:
# (A^2)[1,4] = 2 = the two paths 1->2->4 and 1->3->4. Verify against enumeration.
import numpy as np
from aiq.path_algebra import PathAlgebra
A2 = A @ A
print('A^2 =\n', A2)
print('\n(A^2)[1,4] =', int(A2[0,3]))
print('paths_from_to(1,4,2):', [str(p) for p in PathAlgebra(Q).paths_from_to(1,4,2)])

## 3. Paths are the basis of kQ — and the Proposition

The graded piece `e_j · kQ_g · e_i` has the length-g paths i→j as a basis, so its dimension is `(A^g)[i,j]`.

<img src="../figs-theory/en/T3_paths_basis_kQ.svg" width="760"/>

In [ ]:
# dim(e_4 kQ_g e_1) = (A^g)[1,4] for every g — the Proposition, checked numerically.
for g in range(1, 4):
    dim = len(PathAlgebra(Q).paths_from_to(1, 4, g))
    entry = int(np.linalg.matrix_power(A, g)[0, 3])
    print(f'g={g}:  dim(e4 kQ_{g} e1) = {dim}   (A^{g})[1,4] = {entry}   match: {dim==entry}')

## 4. The grading: kQ = ⊕ kQ_g

Multiplication is concatenation, and it adds lengths — so the algebra splits cleanly by path length.

<img src="../figs-theory/en/T4_grading.svg" width="760"/>

**Next (P3):** each path is a message, so `n_g(i,j)` is how many messages pile up.

---

# P3 — Paths are messages

This connects P1 (over-squashing) and P2 (counting paths). After `g` layers, a message from `i` reaches `j`
**once per length-g path**. So `n_g(i,j) = (A^g)[i,j]` is exactly how many copies of `i`'s information land
at `j` — all forced into one fixed vector.

## 1. Each path delivers one message

<img src="../figs-theory/en/E3a_path_is_message.svg" width="760"/>

## 2. Messages stack into one vector

When the stack exceeds the vector's capacity, the receiver cannot disentangle them — over-squashing.

<img src="../figs-theory/en/E3b_messages_stack.svg" width="760"/>

In [ ]:
# Count raw messages vs the de-duplicated (quotient) count into the target.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
for d in [1,2,3]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'depth {d}: raw {int(raw[d][:,t].sum()):4d}   de-duplicated (kQ/I) {int(eff[d][:,t].sum()):3d}')

## 3. The quotient kQ/I: collapsing equivalent paths

Imposing a relation merges parallel paths into one class, lowering the multiplicity.

<img src="../figs-theory/en/T5_quotient.svg" width="760"/>

In [ ]:
# kQ/I collapses the diamond's two length-2 paths from 2 to 1 (via aiq-quivers).
from oversquash.ideal_bridge import build_quotient_plan
import numpy as np
ei = np.array([[0,0,1,2],[1,2,3,3]])   # 0->1, 0->2, 1->3, 2->3
plan = build_quotient_plan(ei, num_nodes=4, max_length=2)
print('raw multiplicity (0->3, length 2):', plan.raw_mult.get((0,3,2)))
print('effective (kQ/I)                 :', plan.effective_mult.get((0,3,2)))

## 4. The twist: when merging helps, and when it hurts

Collapsing parallel paths only helps when they are truly redundant. In our retrieval task the multiplicity
**is signal**, so collapsing it removes information — a negative result we confirm in the experiments (quotient 0.39 vs raw 0.62 at depth 2; 0.30 vs 0.50 at depth 3, five seeds).

<img src="../figs-theory/en/E3c_signal_vs_noise.svg" width="760"/>

**Next (P4):** keep every path, but *learn* how much to weight each one. That is attention.

---

# P4 — The walk operator is attention

The central idea. A Transformer attends with learned weights over all pairs; aggregating over length-g walks
has the *same shape*, but the path algebra supplies a **sparse, multi-hop support**. Walk Attention keeps the
learned weights of attention and takes the support from the algebra.

## 1. Transformer attention: a learned weighted sum

<img src="../figs-theory/en/E4a_transformer_attention.svg" width="760"/>

## 2. The walk operator has the same shape

<img src="../figs-theory/en/T6_walk_is_attention.svg" width="760"/>

In [ ]:
# The attention support at range g is the nonzero pattern of A^g — sparse (via aiq-quivers).
import torch, numpy as np
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(data.edge_index.numpy(), data.num_nodes, max_length=4)
N = data.num_nodes
for g in range(4):
    nz = int((raw[g] > 0).sum())
    print(f'range g={g+1}: {nz:3d} reachable pairs out of {N*N}  ({100*nz/(N*N):.1f}% of all pairs)')

## 3. The algebra makes attention sparse

Standard attention fills the whole square; the walk-reachable support keeps only the K source→target pairs at the deepest range (1.5–2.6% of all pairs, depending on depth).

<img src="../figs-theory/en/T7_dense_vs_sparse.svg" width="760"/>

## 4. Fixed weights vs learned weights

Same multi-hop support — the only difference is fixed (multiplicity) vs learned (query·key) weights. That
difference is decisive: learned weights can *select* which source matters.

<img src="../figs-theory/en/E4b_fixed_vs_learned.svg" width="760"/>

**Next (P5):** train the three models and watch which solves the task.

---

# P5 — Putting it together

Three models, one long-range retrieval task: **GAT** (1-hop, learned), the **walk operator** (multi-hop,
fixed weights), and **Walk Attention** (multi-hop, learned). The theory predicts the outcome; the code
confirms it.

## 1. Three operators on the same graph

<img src="../figs-theory/en/T8_three_operators.svg" width="760"/>

## 2. Two axes: support and weights

<img src="../figs-theory/en/E5a_support_weights_table.svg" width="760"/>

## 3. Why each model wins or loses

<img src="../figs-theory/en/E5b_why_each.svg" width="760"/>

In [ ]:
# Train the three models on the bottleneck retrieval task (small, fast version).
import torch, torch.nn.functional as F, numpy as np, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model
K, M, DEPTH = 5, 4, 3   # chance = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]

def train_eval(m, tr, va, fwd, epochs=70, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warmup if e<warmup
                                            else 0.5*(1+math.cos(math.pi*(e-warmup)/max(1,epochs-warmup))))
    best = 0.0
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_ = fwd(m,b)
            F.cross_entropy(lg, b.y, ignore_index=-100).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step(); m.eval(); a=[]
        with torch.no_grad():
            for b in va:
                lg,_ = fwd(m,b); mm=b.root_mask
                a.append((lg[mm].argmax(-1)==b.y[mm]).float().mean().item())
        best = max(best, float(np.mean(a)))
    return best

def run(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name=='gat':
        tr=PyGLoader(data[:1200],batch_size=128,shuffle=True); va=PyGLoader(data[1200:],batch_size=128)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None))
    elif name=='walkraw':
        tf=AttachWalkOperators(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_operators)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_operators)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_raw=b.walk_raw)
    else:
        tf=AttachWalkMasks(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_masks)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_masks)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_masks=b.walk_masks)
    m = build_model(name, data[0].x.size(-1), 16, data[0].num_classes, nl, heads=4)
    return train_eval(m, tr, va, fwd)

for name in ['gat','walkraw','walkattn']:
    print(f'{name:10s} final accuracy = {run(name):.3f}')

## 4. The result

GAT degrades toward chance (0.45±0.22 at depth 2, 0.29±0.17 at depth 3 in the full run); the walk operator reaches but cannot select (0.62±0.12 / 0.50±0.12); Walk Attention solves it exactly (1.000±0.000 over 5 seeds at both depths).

<img src="../figs-theory/en/E5c_result.svg" width="760"/>

**The path algebra of a quiver defines a sparse multi-hop attention that mitigates over-squashing.**